In [ ]:
from IPython.core.display import HTML
HTML("""
    <style>
    body { font-feature-settings: "liga" 0; }
    </style>
""")

# Using Antelope modeling tools to customize ecoinvent
At the forefront of the user experience for Antelope is the product model, which is made up of "observed material flows", also known as "fragments". A Fragment is essentially an "exchange"-- where in traditional LCA the "unit process" is the smallest organizing structure for LCA data, in Antelope it is the exchange. The reason for this is simple: every exchange must be defined by a specific data collection process. Each exchange is associated with the movement of a product, material, or substance that can in principle be *observed* by a data collector.  All the information concerning this material flow is associated with the "fragment".

An exchange has five components, three of which can be configuration-dependent [DOI: 10.2139/ssrn.4304957](https://doi.org/10.2139/ssrn.4304957) : 
 1. The reference or "parent" node- this is the activity that is *responsible* for the flow
 2. The flow itself being exchanged, along with its **characteristics**
 3. The direction it is moving relative to the parent node (Input or Output)
 4. The **exchange value** for the flow- the amount required per unit activity of the parent node'
 5. The "other end" of the exchange-- where the flow is coming from. If the flow is coming from outside the *foreground* of the model, it must be **anchored** to some external source.

The modeler must *observe* the exchange value and anchor for each flow to construct an LCA model.

In this demo, we will start with a basic ecoinvent process, and then alter it with new observations that are specific to our situation.  The result of the exercise can be exported as a custom LCI dataset for use in other software.

## Get Set Up

In [ ]:
from antelope_foreground import ForegroundCatalog
from antelope import enum
from antelope_core import ResourceLoader

In [ ]:
cat = ForegroundCatalog()

In [ ]:
with ResourceLoader('/data/LCI/aws-data/') as rl:
    rl.load_resources(cat, 'ecoinvent.3.10.cutoff')
    rl.load_resources(cat, 'lcia.openlca.2.1.4')



In [ ]:
cat.show_interfaces()

## Configure LCIA
We'll use ReCiPe midpoint, egalitarian

In [ ]:
q_lcia = cat.query('lcia.openlca')
lcias = enum(q_lcia.lcia())

In [ ]:
recipe_m_E = lcias[36]
r_cats = enum(q_lcia.get(k) for k in recipe_m_E['ImpactCategories'])

In [ ]:
gwp = r_cats[1]

## Let's build a PET model
polyethylene terephthalate, that is

In [ ]:
q10 = cat.query('ecoinvent.3.10.cutoff')

In [ ]:
pet_flows = enum(q10.flows(name='terephthalate'))

In [ ]:
pet_flows[14].show()

In [ ]:
pets = enum(pet_flows[14].targets())

Perform LCIA

In [ ]:
pet_rer = pets[3]
pet_rer.bg_lcia(gwp).total()

## Create a product system model
This enables us to configure the LCA dynamically

In [ ]:
fg = cat.create_foreground('funny')


In [ ]:
pet_model = fg.create_process_model(pet_rer)

In [ ]:
pet_model.show_tree()

In [ ]:
pet_model.fragment_lcia(gwp).total()

In [ ]:
pet_model.fragment_lcia(gwp).show_components()

We can extract LCI from the product model just as though it were a process

In [ ]:
_=enum(list(pet_model.fragment_lci())[:40])

In [ ]:
gwp.do_lcia(pet_model.fragment_lci()).total()

## Customize the model
The real utility of Antelope is in quickly constructing new product models. One way to do this is by overriding some of the built-in dependencies of the PET production process.

In [ ]:
deps = enum(pet_rer.dependencies())

Pick the exchange we want to modify

In [ ]:
eg_exchange = deps[8]
print(eg_exchange)


What other choices are there for EG production?

In [ ]:
egs=enum(eg_exchange.flow.targets())

We use our "Quick and easy" modeling tool to add a "tap" to the process

In [ ]:
from antelope_reports import QuickAndEasy
mmu = QuickAndEasy(fg)

In [ ]:
eg_tap = mmu.add_tap(pet_model.balance_flow, eg_exchange.flow, 'Input')

By default, this tap acts as a cut-off

In [ ]:
pet_model.show_tree(True)

In [ ]:
pet_model.fragment_lcia(gwp).show_components()

In [ ]:
_=enum(pet_model.cutoffs(True))

The LCI result has changed to reflect the cut-off EG input

In [ ]:
_=enum(list(pet_model.fragment_lci())[:40])

Now we *anchor* the tap (formerly "terminate") to the activity to which it was originally linked

In [ ]:
eg_tap.terminate(q10.get(eg_exchange.termination))

In [ ]:
pet_model.show_tree()

And now the original LCIA score is restored

In [ ]:
pet_model.fragment_lcia(gwp).show_components()

## Change our EG supplier

In [ ]:
_=enum(egs)

We specify an alternative linkage for the flow using a *scenario*.

Again, the modern word for linking an exchange to a provider process is "anchoring" but the legacy word is "terminating" the flow.

In [ ]:
eg_tap.terminate(egs[3], scenario='market for eg [RoW]')

When we specify that scenario, we get alternate results

In [ ]:
pet_model.fragment_lcia(gwp, scenario='market for eg [RoW]').show_components()

In [ ]:
eg_tap.terminate(egs[0], scenario='no market')

In [ ]:
pet_model.fragment_lcia(gwp, scenario='no market').show_components()

In [ ]:
pet_model.show_tree('no market')

## Also customize electricity

### Create a new tap to customize the electricity supply

In [ ]:
elec = deps[10]
print(elec)

In [ ]:
e_tap = mmu.add_tap(pet_model.balance_flow, elec.flow, 'Input', term=q10.get(elec.termination))

In [ ]:
pet_model.show_tree(True)

In [ ]:
pet_model.fragment_lcia(gwp).show_components()

### Override the RER grid with a Chinese grid

In [ ]:
cn = enum(k for k in elec.flow.targets() if k['SpatialScope'].startswith('CN'))

In [ ]:
e_tap.terminate(cn[5], scenario='cn grid')

In [ ]:
pet_model.fragment_lcia(gwp, scenario='cn grid').show_components()

We can compose multiple scenarios together, as long as they do not conflict

In [ ]:
pet_model.fragment_lcia(gwp, scenario=('no market', 'cn grid')).show_components()


We can again extract LCI data for our customized model

In [ ]:
_=enum(list(pet_model.fragment_lci(('no market', 'cn grid')))[:40])